In [1]:
import os
os.environ['PATH']

'C:\\Users\\jiabo\\AppData\\Local\\anaconda3\\envs\\HuggingFace;C:\\Users\\jiabo\\AppData\\Local\\anaconda3\\envs\\HuggingFace\\Library\\mingw-w64\\bin;C:\\Users\\jiabo\\AppData\\Local\\anaconda3\\envs\\HuggingFace\\Library\\usr\\bin;C:\\Users\\jiabo\\AppData\\Local\\anaconda3\\envs\\HuggingFace\\Library\\bin;C:\\Users\\jiabo\\AppData\\Local\\anaconda3\\envs\\HuggingFace\\Scripts;C:\\Users\\jiabo\\AppData\\Local\\anaconda3\\envs\\HuggingFace\\bin;C:\\Users\\jiabo\\AppData\\Local\\anaconda3\\condabin;C:\\Windows\\system32;C:\\Windows;C:\\Windows\\System32\\Wbem;C:\\Windows\\System32\\WindowsPowerShell\\v1.0;C:\\Windows\\System32\\OpenSSH;C:\\Program Files\\Go\\bin;C:\\Strawberry\\c\\bin;C:\\Strawberry\\perl\\site\\bin;C:\\Strawberry\\perl\\bin;C:\\Program Files\\Docker\\Docker\\resources\\bin;C:\\Program Files\\Git\\cmd;C:\\Users\\jiabo\\AppData\\Local\\Microsoft\\WindowsApps;C:\\Users\\jiabo\\AppData\\Local\\Programs\\Microsoft VS Code\\bin;C:\\Users\\jiabo\\AppData\\Local\\anaconda3\\

In [2]:
import logging
from pathlib import Path
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, TrainerCallback
from datasets import load_dataset
from perf_estimator.trainer.plugins import ProfilerCallback, SnapshotCallback

C:\Users\jiabo\AppData\Local\anaconda3\envs\HuggingFace\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AttributeError: module 'numpy' has no attribute 'ndarray'

In [ ]:

os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"


In [3]:
# Configure logging
logging.basicConfig(format='%(asctime)s - %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

# ---------------------------
# 1. Model Initialization (from scratch)
# ---------------------------
# We use the configuration of a popular small-scale LLM (facebook/opt-125m)
# but initialize the model randomly (i.e. train from scratch)
model_name = "facebook/opt-125m"
model_name = "EleutherAI/gpt-neo-125M"
config = AutoConfig.from_pretrained(model_name)  # load config; do NOT load pretrained weights
model = AutoModelForCausalLM.from_config(config)   # randomly initialized model

In [3]:
# Load tokenizer (we can reuse the pretrained tokenizer)
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token  # assign PAD token if missing

# Enable gradient checkpointing to reduce memory usage (at the cost of additional compute)
model.gradient_checkpointing_enable()
logger.info(f"Initialized model from scratch with configuration from '{model_name}'.")

# ---------------------------
# 2. Dataset Preparation
# ---------------------------
# Load the Wikitext-2 dataset as our general-purpose text corpus.
# For a quick profiling run, we use only a small subset.
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
train_dataset = dataset["train"].select(range(1000))  # limit to 1000 examples for this demo

# Tokenization: convert text to token IDs (truncated to a maximum length)
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, max_length=128)

tokenized_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Data collator: handles padding and prepares labels for causal LM (labels equal to input_ids)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ---------------------------
# 3. Trainer Setup with Memory Profiling Callback
# ---------------------------
# TrainingArguments are set to run only 3 steps and use a small batch size suitable for an 8–12GB GPU.
training_args = TrainingArguments(
    output_dir="output",
    per_device_train_batch_size=2,
    max_steps=3,  # run only 3 training iterations for profiling
    gradient_accumulation_steps=1,
    fp16=False,  # using full precision; set True if your GPU supports mixed precision to save memory
    logging_steps=1,
    report_to=[],  # disable external logging (e.g., wandb)
    disable_tqdm=False,
    use_cpu=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    callbacks=[ProfilerCallback(), SnapshotCallback()],
)

# ---------------------------
# 4. Training with PyTorch Profiler
# ---------------------------

trainer.train()  # run 3 training iterations

# Print a summary of the profiler's memory usage by CUDA operation (top 10 ops)
print("Profiler Memory Usage Summary (top CUDA ops):")


2025-03-07 20:09:39,876 - INFO - Initialized model from scratch with configuration from 'EleutherAI/gpt-neo-125M'.


Starting profiler...


/home/glaswigian/miniconda3/envs/Huggingface/lib/python3.12/site-packages/torch/nn/parallel/data_parallel.py:37: UserWarning: 
    There is an imbalance between your GPUs. You may want to exclude GPU 1 which
    has less than 75% of the memory or cores of GPU 0. You can do so by setting
    the device_ids argument to DataParallel, or by setting the CUDA_VISIBLE_DEVICES
    environment variable.
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/home/glaswigian/miniconda3/envs/Huggingface/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
1,5.279300
2,5.157300
3,5.144700


[W307 20:09:47.721222271 CPUAllocator.cpp:245] Memory block of unknown size was allocated before the profiling started, profiler results will not include the deallocation event


Stopping profiler...
Profiler Memory Usage Summary (top CUDA ops):


In [6]:
print(model)

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=False)
            (q_proj): Linear(in_features=768, out_features=768, bias=False)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_fe

In [ ]:
from ures.files import filter_files
p_file = filter_files("pt.trace.json", str(Path().cwd().joinpath("Profiler")), fuzz=True)[-1]


In [8]:
from perf_estimator.profiler import ProfilerDataProcessing
pdp = ProfilerDataProcessing(p_file)

2025-03-10 11:17:16,341 - WARNING - Duplicate layer name found: GPTNeoBlock_11. Renaming to GPTNeoBlock_11_98
2025-03-10 11:17:16,341 - WARNING - Duplicate layer name found: LayerNorm_22. Renaming to LayerNorm_22_f3
2025-03-10 11:17:16,341 - WARNING - Duplicate layer name found: GPTNeoAttention_11. Renaming to GPTNeoAttention_11_af
2025-03-10 11:17:16,341 - WARNING - Duplicate layer name found: GPTNeoSelfAttention_11. Renaming to GPTNeoSelfAttention_11_cd
2025-03-10 11:17:16,341 - WARNING - Duplicate layer name found: Linear_66. Renaming to Linear_66_a6
2025-03-10 11:17:16,341 - WARNING - Duplicate layer name found: Linear_67. Renaming to Linear_67_5e
2025-03-10 11:17:16,341 - WARNING - Duplicate layer name found: Linear_68. Renaming to Linear_68_af
2025-03-10 11:17:16,341 - WARNING - Duplicate layer name found: Dropout_34. Renaming to Dropout_34_ac
2025-03-10 11:17:16,341 - WARNING - Duplicate layer name found: Linear_69. Renaming to Linear_69_c4
2025-03-10 11:17:16,341 - WARNING - Du

In [12]:
layers = pdp.get_iteration(1).get_layers()
layers

{'Embedding_0': <perf_estimator.profiler.analyser.Layer at 0x19c9c0268d0>,
 'Embedding_1': <perf_estimator.profiler.analyser.Layer at 0x19c9c0269d0>,
 'Dropout_0': <perf_estimator.profiler.analyser.Layer at 0x19c9c026ad0>,
 'LayerNorm_0': <perf_estimator.profiler.analyser.Layer at 0x19c9c026cd0>,
 'Linear_0': <perf_estimator.profiler.analyser.Layer at 0x19c9c026fd0>,
 'Linear_1': <perf_estimator.profiler.analyser.Layer at 0x19c9c0270d0>,
 'Linear_2': <perf_estimator.profiler.analyser.Layer at 0x19c9c0271d0>,
 'Dropout_1': <perf_estimator.profiler.analyser.Layer at 0x19c9c0272d0>,
 'Linear_3': <perf_estimator.profiler.analyser.Layer at 0x19c9c0273d0>,
 'Dropout_2': <perf_estimator.profiler.analyser.Layer at 0x19c9c0274d0>,
 'LayerNorm_1': <perf_estimator.profiler.analyser.Layer at 0x19c9c0275d0>,
 'Linear_4': <perf_estimator.profiler.analyser.Layer at 0x19c9c0277d0>,
 'NewGELUActivation_0': <perf_estimator.profiler.analyser.Layer at 0x19c9c0278d0>,
 'Linear_5': <perf_estimator.profiler.

In [16]:
count = 0
for name, layer in layers.items():
    print(name)
    if "GPTNeoBlock" in name:
        break
count

Embedding_0
Embedding_1
Dropout_0
LayerNorm_0
Linear_0
Linear_1
Linear_2
Dropout_1
Linear_3
Dropout_2
LayerNorm_1
Linear_4
NewGELUActivation_0
Linear_5
Dropout_3
LayerNorm_2
Linear_6
Linear_7
Linear_8
Dropout_4
Linear_9
Dropout_5
LayerNorm_3
Linear_10
NewGELUActivation_1
Linear_11
Dropout_6
LayerNorm_4
Linear_12
Linear_13
Linear_14
Dropout_7
Linear_15
Dropout_8
LayerNorm_5
Linear_16
NewGELUActivation_2
Linear_17
Dropout_9
LayerNorm_6
Linear_18
Linear_19
Linear_20
Dropout_10
Linear_21
Dropout_11
LayerNorm_7
Linear_22
NewGELUActivation_3
Linear_23
Dropout_12
LayerNorm_8
Linear_24
Linear_25
Linear_26
Dropout_13
Linear_27
Dropout_14
LayerNorm_9
Linear_28
NewGELUActivation_4
Linear_29
Dropout_15
LayerNorm_10
Linear_30
Linear_31
Linear_32
Dropout_16
Linear_33
Dropout_17
LayerNorm_11
Linear_34
NewGELUActivation_5
Linear_35
Dropout_18
LayerNorm_12
Linear_36
Linear_37
Linear_38
Dropout_19
Linear_39
Dropout_20
LayerNorm_13
Linear_40
NewGELUActivation_6
Linear_41
Dropout_21
LayerNorm_14
Linear_42

0

In [15]:
pdp.get_iteration(1).plot_memory_change()

AttributeError: 'IterationData' object has no attribute 'plot_memory_change'

In [ ]:
from perf_estimator.estimator import Estimator, TrainerEstimator
